# AI Hub 해안쓰레기 탐지 모델 — Colab 이어서 학습

준비: Google Drive 에 `aerodrone/` 폴더를 만들고 `seg_aihub_gsd.zip`(1.6 GB) 과 `aihub_gsd_last.pt` 를 올려 둔다.
런타임 → GPU(A100 또는 L4) 선택 후 위에서부터 실행.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip -q install ultralytics==8.4.* 2>&1 | tail -1

In [ ]:
# 데이터 풀기 (로컬 디스크로 — Drive 에서 바로 읽으면 느림)
import os, re
D = '/content/drive/MyDrive/aerodrone'
!mkdir -p /content/data && tar -xf $D/seg_aihub_gsd.zip -C /content/data   # 이름은 .zip이지만 tar 형식
# data.yaml 의 경로를 Colab 경로로 바꿈
y = open('/content/data/seg_aihub_gsd/data.yaml', encoding='utf-8').read()
y = re.sub(r'^path:.*$', 'path: /content/data/seg_aihub_gsd', y, flags=re.M)
open('/content/data/seg_aihub_gsd/data.yaml', 'w', encoding='utf-8').write(y)
print(y)

In [ ]:
# 이어서 학습 — 노트북(8 GB)에서 5/12 에폭까지 한 last.pt 에서 시작.
# Colab 큰 GPU 이점: 에폭을 늘리고(30) 배치를 키움(32). 시간 여유 없으면 epochs=15.
from ultralytics import YOLO
m = YOLO(f'{D}/aihub_gsd_last.pt')
m.train(data='/content/data/seg_aihub_gsd/data.yaml', epochs=30, imgsz=1024, batch=32,
        project='/content/runs', name='aihub_gsd_colab', exist_ok=True,
        patience=8, cos_lr=True, close_mosaic=5, workers=8, cache='ram')

In [ ]:
# 결과를 Drive 로 (best.pt 를 노트북 runs/seg/aihub_gsd_det_s/weights/ 에 받아 쓰면 됨)
!mkdir -p $D/out && cp /content/runs/aihub_gsd_colab/weights/best.pt $D/out/aihub_gsd_colab_best.pt
!cp /content/runs/aihub_gsd_colab/results.csv $D/out/ && cp /content/runs/aihub_gsd_colab/results.png $D/out/ 2>/dev/null
import pandas as pd; r = pd.read_csv('/content/runs/aihub_gsd_colab/results.csv'); r.columns = r.columns.str.strip()
print(r[['epoch', 'metrics/mAP50(B)', 'metrics/mAP50-95(B)']].tail(5))

## 새 가중치로 세 데이터셋 교차평가 (문갑도 업체 칩 · 하와이 해안 · 튀니지 해변)
Drive 에 `evalpack.zip` 과 `eval_models.py` 도 올려 두어야 한다. 결과 표는 `aerodrone/out/cross_eval.md`.

In [ ]:
!tar -xf $D/evalpack.zip -C /content   # tar 형식
!python $D/eval_models.py --pack /content/evalpack --out $D/out \n    --weights 5에폭:$D/aihub_gsd_last.pt 콜랩학습:/content/runs/aihub_gsd_colab/weights/best.pt \n    --world yolov8s-worldv2.pt --max 400